# Student Information

**Full Name:** [Your Full Name]

**Student ID:** 21201063

**Section:** [Your Section]

# Import Libraries

In [1]:
# Import required libraries
import random

print("Libraries imported successfully.")

Libraries imported successfully.


# Player Class

In [2]:
# Player class - stores player name, role and alive status

class Player:

    def __init__(self, name, role):
        self.name = name
        self.role = role
        self.alive = True

    def display_info(self):
        # check if player is alive or dead
        status = "Alive"
        if self.alive == False:
            status = "Eliminated"
        print(self.name + " - " + self.role + " (" + status + ")")

    def eliminate(self):
        # mark player as dead
        self.alive = False
        print(self.name + " has been eliminated.")

# Game Class

In [3]:
# Game class - controls the whole game

class Game:

    def __init__(self):
        self.players = []
        self.round_number = 0
        self.eliminated_order = []

    # load player names from players.txt file
    def load_players(self, filename):
        print("Loading players from " + filename + "...")

        file = open(filename, "r")
        for line in file:
            name = line.strip()
            if name != "":
                # create a player object for each name
                player = Player(name, "Unassigned")
                self.players.append(player)
        file.close()

        print("Loaded " + str(len(self.players)) + " players.")

    # randomly assign 2 werewolves, rest are villagers
    def assign_roles(self):
        print("\nAssigning roles randomly...")

        # shuffle the list then assign roles
        shuffled = list(self.players)
        random.shuffle(shuffled)

        for i in range(len(shuffled)):
            if i < 2:
                shuffled[i].role = "Werewolf"
            else:
                shuffled[i].role = "Villager"

        print("Roles have been assigned. (Shh... it's a secret!)")

    # get list of alive players
    def get_alive_players(self):
        alive_list = []
        for player in self.players:
            if player.alive == True:
                alive_list.append(player)
        return alive_list

    # count alive players with a specific role
    def count_alive_by_role(self, role):
        count = 0
        for player in self.players:
            if player.alive == True and player.role == role:
                count = count + 1
        return count

    # show all alive players
    def show_alive_players(self):
        alive = self.get_alive_players()
        print("\nAlive Players (" + str(len(alive)) + "):")
        for player in alive:
            print("  - " + player.name)

    # night phase - werewolves attack someone
    def night_phase(self):

        print("\n-- Night " + str(self.round_number) + " --")
        print("The village sleeps...")

        # find alive werewolves
        alive_wolves = []
        for player in self.players:
            if player.alive == True and player.role == "Werewolf":
                alive_wolves.append(player)

        if len(alive_wolves) == 0:
            print("No werewolves are alive. The night is peaceful.")
            return

        # find alive villagers
        alive_villagers = []
        for player in self.players:
            if player.alive == True and player.role == "Villager":
                alive_villagers.append(player)

        if len(alive_villagers) == 0:
            print("No villagers left to attack.")
            return

        # pick a random villager
        target = random.choice(alive_villagers)

        # check for lucky escape - 10% chance
        lucky_chance = random.randint(1, 100)
        if lucky_chance <= 10:
            print("\n*** Special Event: Lucky Escape! ***")
            print("The target escaped from the Werewolves.")
            print("Nobody dies tonight.")
        else:
            # normal attack
            print("\nWerewolves attacked " + target.name + ".")
            target.eliminate()
            self.eliminated_order.append(target.name)

    # day phase - villagers vote someone out
    def day_phase(self):

        print("\n-- Day " + str(self.round_number) + " --")
        print("The village discusses...")

        alive = self.get_alive_players()

        if len(alive) <= 1:
            print("Not enough players for a vote.")
            return

        # check for mysterious clue - 15% chance
        clue_chance = random.randint(1, 100)
        if clue_chance <= 15:
            # find alive werewolves
            alive_wolves = []
            for player in alive:
                if player.role == "Werewolf":
                    alive_wolves.append(player)

            if len(alive_wolves) > 0:
                # eliminate a werewolf directly
                caught_wolf = random.choice(alive_wolves)
                print("\n*** Special Event: Mysterious Clue! ***")
                print("The villagers found evidence and eliminated a Werewolf.")
                print("Villagers identified " + caught_wolf.name + " as a Werewolf.")
                caught_wolf.eliminate()
                self.eliminated_order.append(caught_wolf.name)
                return

        # normal voting - pick someone randomly
        suspect = random.choice(alive)
        print("\nVillagers voted against " + suspect.name + ".")
        print(suspect.name + " was eliminated.")
        suspect.eliminate()
        self.eliminated_order.append(suspect.name)

    # check if someone won
    def check_winner(self):

        wolf_count = self.count_alive_by_role("Werewolf")
        villager_count = self.count_alive_by_role("Villager")

        # villagers win when both wolves are dead
        if wolf_count == 0:
            return "Villagers"

        # wolves win when they equal or outnumber villagers
        if wolf_count >= villager_count:
            return "Werewolves"

        # no winner yet
        return None

    # save game result to file
    def save_result(self, winner):

        print("\nSaving results to game_result.txt...")

        alive = self.get_alive_players()
        surviving_names = []
        for player in alive:
            surviving_names.append(player.name)

        # write to file
        file = open("game_result.txt", "w")
        file.write("=== WEREWOLF GAME RESULT ===\n")
        file.write("Winner: " + winner + "\n")
        file.write("Total Rounds: " + str(self.round_number) + "\n")
        file.write("\n--- Surviving Players ---\n")
        for name in surviving_names:
            file.write(name + " (" + self.get_role_by_name(name) + ")\n")
        file.write("\n--- Eliminated Players (in order) ---\n")
        for name in self.eliminated_order:
            file.write(name + "\n")
        file.close()

        print("Results saved successfully!")

    # get role of a player by name
    def get_role_by_name(self, name):
        for player in self.players:
            if player.name == name:
                return player.role
        return "Unknown"

    # show current villager and werewolf count
    def print_status(self):
        v_count = self.count_alive_by_role("Villager")
        w_count = self.count_alive_by_role("Werewolf")
        print("\nCurrent Status:")
        print("  Villagers: " + str(v_count))
        print("  Werewolves: " + str(w_count))

    # main game loop
    def start_game(self):

        print("\n" + "-" * 30)
        print("WEREWOLF GAME SIMULATOR")
        print("-" * 30)

        winner = None

        # keep playing until someone wins
        while winner is None:
            self.round_number = self.round_number + 1

            print("\n--- Round " + str(self.round_number) + " ---")

            self.show_alive_players()
            self.night_phase()

            # check after night
            winner = self.check_winner()
            if winner is not None:
                break

            self.day_phase()
            self.print_status()

            # check after day
            winner = self.check_winner()

        # game over
        print("\n" + "-" * 30)
        print("GAME OVER!")
        print("Winner: " + winner.upper())
        print("-" * 30)

        self.save_result(winner)

        # show everyone's role
        print("\n--- Final Role Reveal ---")
        for player in self.players:
            player.display_info()

# Create Players File

In [4]:
# creating players.txt with 8 names

player_names = [
    "Rafiq",
    "Fatima",
    "Hasan",
    "Nasrin",
    "Kamal",
    "Ayesha",
    "Jahid",
    "Shamima"
]

# write each name on a new line
file = open("players.txt", "w")
for name in player_names:
    file.write(name + "\n")
file.close()

print("players.txt has been created with " + str(len(player_names)) + " names.")

players.txt has been created with 8 names.


# Run Game Simulation

In [5]:
# running the game

game = Game()

game.load_players("players.txt")
game.assign_roles()
game.start_game()

Loading players from players.txt...
Loaded 8 players.

Assigning roles randomly...
Roles have been assigned. (Shh... it's a secret!)

------------------------------
WEREWOLF GAME SIMULATOR
------------------------------

--- Round 1 ---

Alive Players (8):
  - Rafiq
  - Fatima
  - Hasan
  - Nasrin
  - Kamal
  - Ayesha
  - Jahid
  - Shamima

-- Night 1 --
The village sleeps...

Werewolves attacked Hasan.
Hasan has been eliminated.

-- Day 1 --
The village discusses...

Villagers voted against Ayesha.
Ayesha was eliminated.
Ayesha has been eliminated.

Current Status:
  Villagers: 5
  Werewolves: 1

--- Round 2 ---

Alive Players (6):
  - Rafiq
  - Fatima
  - Nasrin
  - Kamal
  - Jahid
  - Shamima

-- Night 2 --
The village sleeps...

Werewolves attacked Jahid.
Jahid has been eliminated.

-- Day 2 --
The village discusses...

Villagers voted against Shamima.
Shamima was eliminated.
Shamima has been eliminated.

Current Status:
  Villagers: 3
  Werewolves: 1

--- Round 3 ---

Alive Player

# Result File Output

In [6]:
# showing the result file

print("-" * 30)
print("CONTENTS OF game_result.txt")
print("-" * 30)

result_file = open("game_result.txt", "r")
for line in result_file:
    print(line, end="")
result_file.close()

print("-" * 30)

------------------------------
CONTENTS OF game_result.txt
------------------------------
=== WEREWOLF GAME RESULT ===
Winner: Werewolves
Total Rounds: 3

--- Surviving Players ---
Fatima (Werewolf)
Kamal (Villager)

--- Eliminated Players (in order) ---
Hasan
Ayesha
Jahid
Shamima
Nasrin
Rafiq
------------------------------


# Reflection Section

## Question 1: Which part was the hardest?

The hardest part for me was figuring out the game-ending condition. At first, I did not realize that Werewolves win when their number equals or exceeds the Villagers. I kept checking if all villagers were dead, but that is not the same thing. I had to read the assignment requirements carefully a few times. Once I understood the rule, writing the `check_winner()` method was straightforward.

## Question 2: What bug/problem did you face?

I ran into a problem where the game would crash during the night phase if no werewolves were alive. My code tried to pick a random villager to attack, but it also assumed there was always a werewolf making the attack. I fixed this by adding a check at the start of the `night_phase()` method. Now if there are no alive werewolves, the night phase just prints a message and returns safely.

## Question 3: What did you learn from this assignment?

This assignment taught me how to use classes in Python for a real project. Before this, I only wrote small programs without classes. I learned how to separate different parts of a program into methods. I also learned about file handling — reading from `players.txt` and writing results to `game_result.txt`. The random module was fun to use for picking targets and suspects. Overall, I now feel more comfortable with object-oriented programming.

## Question 4: What would you improve later?

If I had more time, I would add more roles to the game, like a Doctor who can save one person each night or a Seer who can check someone's role. I would also make the game interactive so the user can type commands instead of just watching the simulation run. Another improvement would be adding a graphical interface using a library like `tkinter`. I think these additions would make the project more interesting and fun to play.